# Step 06 — Resume Parser: post evaluation on Kaggle

Use a fresh **GPU T4 x2** session with Internet on, then **Save Version → Save & Run All**.
This notebook loads the completed adapter published to `md-zain/resume-parser` from public `main`, resolved once to an immutable
commit for this run. It does **no training**, validation-set tuning, or model publication.

The existing Step 04 evaluator checks JSON/schema validity, deterministic fields, and TypeSafe AI
semantic matches. TypeSafe receives resume evidence and expected/generated semantic values,
as in the existing evaluator. Enable `TYPESAFE_API_KEY` and `WANDB_API_KEY` in Kaggle Secrets. `HF_TOKEN` is optional
for the public model/dataset; enable `GITHUB_TOKEN` if GitHub source is private.

Both T4s evaluate disjoint held-out rows with independent 4-bit/FP16 model replicas. Every reserved evaluation
row is attempted, with the compact training prompt and a fixed **2,048 output-token ceiling**.
Output-limit hits, invalid JSON, context errors and semantic-service errors remain visible in the results.
Scores are benchmark measurements, not a claim of universal accuracy.

Anonymous scores are saved after **each resume** under `/kaggle/working/post-evaluation`.
If interrupted, attach that output as Kaggle input and set `PREVIOUS_RESULTS_DIR` in Cell 6.
Completed cases are reused only when the model, data, evaluator and settings match.
Failures caused by inference/semantic services are retried on the next run; invalid model outputs are retained.
Raw resumes, targets, model outputs, credentials and caches stay under `/tmp`, outside saved outputs.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# Cell 1 — Expose both Kaggle T4s before importing the inference stack.
import subprocess
import sys
import time  # Used only for network retries, not an evaluation limit.
from pathlib import Path

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook in Kaggle with a GPU accelerator enabled.')
if any(name in sys.modules for name in ('torch', 'unsloth', 'transformers', 'peft', 'trl')):
    raise RuntimeError('Start a fresh Kaggle session and run from the first cell before importing the inference stack.')
if int(os.getenv('WORLD_SIZE', '1')) != 1:
    raise RuntimeError('Use the normal single-process notebook kernel, not a distributed launcher.')
PRIVATE_ROOT = Path('/tmp/resume-parser-post-evaluation')
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)
PRIVATE_ROOT.chmod(0o700)
os.environ['HF_HOME'] = str(PRIVATE_ROOT / 'huggingface')
os.environ['HF_TOKEN_PATH'] = str(PRIVATE_ROOT / 'huggingface/token')
os.environ['WANDB_DIR'] = str(PRIVATE_ROOT / 'wandb')
Path(os.environ['WANDB_DIR']).mkdir(parents=True, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# nvidia-smi lists physical devices; the child-process check confirms CUDA visibility.
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-c', (
    'import torch; '
    'print("torch.cuda.device_count():", torch.cuda.device_count()); '
    'assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, '
    '"Expected visible CUDA GPUs; start a fresh GPU session."; '
    'print("Visible GPU:", torch.cuda.get_device_name(0))'
)], check=True)

In [ ]:
# Cell 2 — Load Kaggle Secrets without displaying credentials.
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
ENV_VARIABLE_NAMES = [
    'HF_TOKEN', 'TYPESAFE_API_KEY', 'WANDB_API_KEY', 'GITHUB_TOKEN',
    'WANDB_ENTITY', 'WANDB_PROJECT', 'HF_DATASET_REVISION',
]
loaded_secret_names = set()
for name in ENV_VARIABLE_NAMES:
    try:
        value = s.get_secret(name)
    except Exception:
        value = None
    if value and value.strip():
        os.environ[name] = value.strip()
        loaded_secret_names.add(name)
required = {'TYPESAFE_API_KEY', 'WANDB_API_KEY'}
if not required <= loaded_secret_names:
    raise RuntimeError('Enable these Kaggle Secrets: ' + ', '.join(sorted(required - loaded_secret_names)))
hf_token = os.getenv('HF_TOKEN') or False  # Public model/dataset also work anonymously.
print('Enabled secret names:', ', '.join(sorted(loaded_secret_names)))

In [ ]:
# Cell 3 — Clone public/private GitHub source, or copy an attached Kaggle source folder.
import base64
import shutil
import socket
from urllib.parse import urlsplit

REPO_URL = 'https://github.com/Muhammad-Zain01/resume-parser-model.git'  # [REPO_URL]
SOURCE_DIR = None  # Optional: '/kaggle/input/YOUR-SOURCE-DATASET/resume-parser-model'
PROJECT_ROOT = PRIVATE_ROOT / 'source'
MARKER = Path('src/steps/05_fine_tuning/fine_tuning.py')

def check_dns(host):
    for attempt in range(3):
        try:
            socket.getaddrinfo(host, 443)
            return
        except socket.gaierror:
            if attempt < 2:
                time.sleep(2)
    raise RuntimeError(
        f'Cannot resolve {host}. Enable Internet in Kaggle notebook settings and save those '
        'settings before Save & Run All. If Internet is already enabled, restart the session '
        'and retry; this is a network/DNS problem. For GitHub-only outages, attach a source '
        'folder as Kaggle input and set SOURCE_DIR above. Pip and Hugging Face still need Internet.'
    )

if SOURCE_DIR:
    source = Path(SOURCE_DIR).resolve()
    if not (source / MARKER).is_file():
        raise FileNotFoundError('SOURCE_DIR must contain src/steps/05_fine_tuning/fine_tuning.py.')
    if PROJECT_ROOT.exists():
        raise RuntimeError('Source is already staged. Start a fresh session before changing SOURCE_DIR.')
    shutil.copytree(source, PROJECT_ROOT, ignore=shutil.ignore_patterns(
        '.git', '.env', '.env.*', '.data', '.venv', '__pycache__', '*.ipynb_checkpoints',
    ))
else:
    parsed = urlsplit(REPO_URL)
    if (parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username
            or parsed.password or parsed.query or parsed.fragment or parsed.port):
        raise ValueError('Use a clean https://github.com/OWNER/REPO.git URL without credentials or query parameters.')
    check_dns('github.com')
    git_env = os.environ.copy()
    git_env['GIT_TERMINAL_PROMPT'] = '0'
    github_token = os.getenv('GITHUB_TOKEN', '').strip()
    encoded_auth = ''
    if github_token:
        encoded_auth = base64.b64encode(('x-access-token:' + github_token).encode()).decode()
        # Supply authorization only to GitHub via a child-process environment, never the URL/config file.
        git_env['GIT_CONFIG_COUNT'] = '1'
        git_env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
        git_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + encoded_auth
    def run_git(arguments):
        result = subprocess.run(['git', *arguments], env=git_env, capture_output=True, text=True)
        if result.returncode:
            details = result.stderr
            for secret in (github_token, encoded_auth):
                if secret:
                    details = details.replace(secret, '[redacted]')
            if 'Could not resolve host' in details:
                raise RuntimeError('GitHub DNS lookup failed. Enable Kaggle Internet or use SOURCE_DIR.\n' + details) from None
            raise RuntimeError(
                'Git failed. For a private repository, enable GITHUB_TOKEN with read access to this repository.\n'
                + details
            ) from None
        return result.stdout.strip()
    try:
        if (PROJECT_ROOT / '.git').is_dir():
            if run_git(['-C', str(PROJECT_ROOT), 'remote', 'get-url', 'origin']) != REPO_URL:
                raise RuntimeError('Existing checkout uses another repository. Start a fresh session.')
            run_git(['-C', str(PROJECT_ROOT), 'pull', '--ff-only'])
        elif PROJECT_ROOT.exists():
            raise RuntimeError('Incomplete source folder exists. Restart the Kaggle session before retrying the clone.')
        else:
            run_git(['clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)])
    finally:
        # The token is not retained in a Git URL, command argument, or authorization config on disk.
        git_env.clear()
        github_token = encoded_auth = ''
if not (PROJECT_ROOT / MARKER).is_file():
    raise RuntimeError('The repository does not contain the expected Step 05 module used for the compact prompt.')
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
# Create .env in the private /tmp checkout, outside Kaggle saved outputs.
ENV_FILE = PROJECT_ROOT / '.env'
def dotenv_value(value):
    return "'" + value.replace('\\', '\\\\').replace("'", "\\'") + "'"
with ENV_FILE.open('w', encoding='utf-8') as handle:
    os.chmod(ENV_FILE, 0o600)
    for name in sorted(loaded_secret_names):
        if name in ENV_VARIABLE_NAMES:
            handle.write(name + '=' + dotenv_value(os.environ[name]) + '\n')
print('Private environment file created; values are hidden.')

In [ ]:
# Cell 4 — Install the same working Unsloth dependency profile without replacing Kaggle's Torch/CUDA packages.
import subprocess
import sys
from collections import deque
from importlib.metadata import version, PackageNotFoundError

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this GPU dependency installation in Kaggle, not on the local computer.')
if any(name in sys.modules for name in ('torch', 'unsloth', 'transformers', 'peft', 'trl')):
    raise RuntimeError('Restart the Kaggle session before installing packages into an already imported training stack.')
preserved_versions = {}
for name in ('torch', 'torchvision', 'torchaudio', 'triton'):
    try:
        preserved_versions[name] = version(name)
    except PackageNotFoundError:
        if name == 'torch':
            raise RuntimeError('Select a Kaggle GPU image with CUDA Torch preinstalled.') from None
preserved_pins = [f'{name}=={release}' for name, release in preserved_versions.items()]
# The resolver must honor these pins; never use --no-deps or force a CUDA stack upgrade.
check_dns('pypi.org')

# Published Unsloth 2026.10.3 requires TRL <=1.13.0.
# TRL 1.13.0 requires Datasets >=4.7.0; Unsloth requires Datasets <5.
KAGGLE_PACKAGES = [
    'unsloth==2026.10.3',
    'unsloth_zoo==2026.10.6',
    'transformers==5.17.0',
    'trl==1.13.0',
    'datasets==4.7.0',
    'accelerate==1.15.0',
    'peft==0.21.2',
    'bitsandbytes==0.50.2',
    'wandb==0.30.0',
    'python-dotenv>=1.0.0',
    'pydantic>=2.7.0',
    'tqdm>=4.66.0',
    'matplotlib>=3.8.0',
    'typesafe-sdk>=0.7.0',
]
# Stream pip output through the notebook and include its final lines in any failure.
# CalledProcessError alone only reports an exit status; it does not explain the cause.
print('Python:', sys.version.split()[0], '| executable:', sys.executable, flush=True)
print('Preserving installed CUDA stack:', preserved_versions, flush=True)
pip_command = [
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '--progress-bar', 'off',
    '--upgrade-strategy', 'only-if-needed', *preserved_pins, *KAGGLE_PACKAGES,
]
pip_tail = deque(maxlen=80)
with subprocess.Popen(
    pip_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
        pip_tail.append(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(
        f'Kaggle dependency installation failed (pip exit code {return_code}). '
        'Do not run the model/evaluation cells. Share the ERROR lines below so the '
        'failing package or dependency conflict can be identified.\n\n'
        + ''.join(pip_tail)
    )
for name, previous in preserved_versions.items():
    if version(name) != previous:
        raise RuntimeError(f'{name} changed unexpectedly. Start a fresh Kaggle GPU session.')
print('Dependency installation completed; existing CUDA package versions were preserved.', flush=True)
# If resolution fails, keep the printed ERROR lines; do not layer arbitrary Torch upgrades.
# Do not install root requirements.txt afterward: its training pins differ from this profile.
# Installation guide: https://unsloth.ai/docs/get-started/install/pip-install

In [ ]:
# Cell 5 — Import Unsloth first and stage the runner bundled with this notebook.
from unsloth import FastLanguageModel
import torch
import json
import hashlib
from importlib import import_module
from importlib.util import module_from_spec, spec_from_file_location
from importlib.metadata import version
from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download
from transformers import AutoConfig

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Bundled so uploading this notebook is sufficient; no separate runner upload is needed.
runner_source = r'''
"""Kaggle post-evaluation inference and anonymous, resumable Step 04 scoring.

No training or model publication is performed. Importing this module does not
load the model, read datasets, or contact TypeSafe.
"""
from __future__ import annotations

import argparse
from importlib import import_module
import json
import os
from pathlib import Path
import sys
import time


def atomic_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)


def load_case(path: Path, fingerprint: str) -> dict:
    saved = json.loads(path.read_text(encoding='utf-8'))
    if saved['fingerprint'] != fingerprint:
        raise ValueError('Saved scores belong to another model/dataset/configuration.')
    return saved


def make_callback(model, tokenizer, job: dict, fine_tuning, evaluation):
    import torch
    from transformers import GenerationConfig, StoppingCriteria, StoppingCriteriaList

    class CompleteJson(StoppingCriteria):
        def __init__(self, start):
            self.start = start
            self.complete = False

        def __call__(self, input_ids, scores, **kwargs):
            # Check periodically; do not keep generating after the first complete JSON object.
            if (input_ids.shape[1] - self.start) % 16:
                return False
            text = tokenizer.decode(input_ids[0, self.start:], skip_special_tokens=True)
            try:
                self.complete = isinstance(json.loads(evaluation.extract_json_object(text)), dict)
            except (ValueError, TypeError):
                self.complete = False
            return self.complete

    def callback(item):
        prompt = fine_tuning.render_training_prompt(item.resume_text, style='compact')
        text = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to(model.device)
        prompt_tokens = inputs['input_ids'].shape[1]
        budget = min(job['max_new_tokens'], job['context_length'] - prompt_tokens)
        if budget < 1:
            raise ValueError('Input exceeds the native context; this case is counted as a failure.')
        stop = CompleteJson(prompt_tokens)
        config = GenerationConfig(
            max_new_tokens=budget, do_sample=False, use_cache=True,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
        with torch.inference_mode():
            output = model.generate(
                **inputs, generation_config=config,
                stopping_criteria=StoppingCriteriaList([stop]),
            )
        generated = output[0, prompt_tokens:]
        ended_with_eos = bool(len(generated) and int(generated[-1]) == tokenizer.eos_token_id)
        return evaluation.ModelCallbackOutput(
            raw_output=tokenizer.decode(generated, skip_special_tokens=True).strip(),
            model_name=f"{job['model_repo']}@{job['model_sha']}",
            metadata={
                'prompt_tokens': prompt_tokens, 'generated_tokens': len(generated),
                'generation_budget': budget,
                'hit_output_limit': len(generated) >= budget and not ended_with_eos and not stop.complete,
                'stopped_on_complete_json': stop.complete,
            },
        )
    return callback


RETRYABLE = {'CALLBACK_FAILED', 'SCORED_WITH_SEMANTIC_ERROR', 'SEMANTIC_JUDGE_FAILED'}
SCORE_FIELDS = (
    'status', 'json_valid', 'schema_valid', 'programmatic_score', 'semantic_score',
    'overall_score', 'identity_score', 'duration_seconds', 'model_metadata',
)
BREAKDOWN_FIELDS = {
    'path', 'group', 'metric', 'value', 'precision', 'recall', 'f1',
    'true_positive', 'false_positive', 'false_negative', 'exact_cases', 'jev_cases', 'jev_score',
}


def worker(job: dict, rank: int, world_size: int) -> None:
    # The notebook assigns one physical T4 to each child before CUDA imports.
    from unsloth import FastLanguageModel
    import torch
    root = Path(job['project_root'])
    sys.path.insert(0, str(root))
    fine_tuning = import_module('src.steps.05_fine_tuning.fine_tuning')
    evaluation = import_module('src.steps.04_evaluation')
    if not os.getenv('TYPESAFE_API_KEY'):
        raise RuntimeError('Enable TYPESAFE_API_KEY before post evaluation.')
    if torch.cuda.device_count() != 1:
        raise RuntimeError('Each evaluation worker must see exactly one CUDA GPU.')
    torch.manual_seed(42)
    records = fine_tuning.load_records(job['evaluation_path'])
    output = Path(job['output_dir'])
    pending = []
    for index in range(rank, len(records), world_size):
        path = output / 'cases' / f'case-{index + 1:05d}.json'
        if path.exists() and load_case(path, job['fingerprint'])['status'] not in RETRYABLE:
            continue
        pending.append((index, path))
    print(f'GPU worker {rank}: {len(pending)} cases remaining.', flush=True)
    if not pending:
        return
    # Unsloth reads adapter_config.json, loads its quantized base, and attaches
    # the existing trained adapter. Do not create a fresh LoRA adapter here.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=job['adapter_path'], max_seq_length=job['context_length'],
        dtype=torch.float16, load_in_4bit=True, device_map={'': 0},
        token=os.getenv('HF_TOKEN') or None,
    )
    if not getattr(model, 'peft_config', None):
        raise RuntimeError('Refusing to evaluate a base model without the trained adapter.')
    if not getattr(model, 'is_loaded_in_4bit', False):
        raise RuntimeError('The evaluation model must be loaded in 4-bit.')
    if not tokenizer.chat_template or tokenizer.eos_token_id is None:
        raise RuntimeError('The adapter tokenizer requires its chat template and EOS token.')
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.requires_grad_(False)
    model.eval()
    FastLanguageModel.for_inference(model)
    print(f"GPU {rank}: loaded {job['model_repo']}@{job['model_sha']} | "
          'trained QLoRA adapter attached | 4-bit | FP16 | inference ready.', flush=True)
    torch.cuda.reset_peak_memory_stats()
    callback = make_callback(model, tokenizer, job, fine_tuning, evaluation)
    consecutive_judge_errors = 0
    started = time.monotonic()
    for finished, (index, path) in enumerate(pending, 1):
        record = records[index]
        case = f'case-{index + 1:05d}'
        item = evaluation.EvaluationItem(
            resume_id=case, resume_text=record['resume_text'], target_json=record['target_json'],
            category=record.get('category'),
        )
        result = evaluation.Evaluator([item]).evaluate(
            callback, workers=1, test_size=None, show_progress=False, output_dir=None,
        ).results[0]
        saved = {
            'fingerprint': job['fingerprint'], 'case': case, 'category': item.category,
            **{name: getattr(result, name) for name in SCORE_FIELDS},
            'score_breakdown': [
                {key: value for key, value in entry.items() if key in BREAKDOWN_FIELDS}
                for entry in result.score_breakdown
            ],
        }
        # Never save prompts, targets, predictions, identity fields, or raw API errors.
        atomic_json(path, saved)
        print(f'GPU {rank}: {finished}/{len(pending)} | {case} | {result.status} | '
              f'{result.duration_seconds:.1f}s | output limit: '
              f'{bool(result.model_metadata.get("hit_output_limit"))}', flush=True)
        consecutive_judge_errors = (
            consecutive_judge_errors + 1 if result.status in
            {'SCORED_WITH_SEMANTIC_ERROR', 'SEMANTIC_JUDGE_FAILED'} else 0
        )
        if consecutive_judge_errors >= 3:
            print('TypeSafe scoring failed three times consecutively. Check TYPESAFE_API_KEY '
                  'and TypeSafe quota before resuming.', flush=True)
            raise RuntimeError('Three consecutive TypeSafe scoring errors. Scores are saved; check the key/quota before resuming.')
    atomic_json(output / f'gpu-{rank}.json', {
        'gpu': rank, 'allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
        'reserved_gib': torch.cuda.max_memory_reserved() / 2**30,
        'processed_this_session': len(pending), 'elapsed_seconds': time.monotonic() - started,
    })


def summarize(job: dict):
    """Rebuild the existing evaluator report from anonymous per-case score files."""
    import pandas as pd
    sys.path.insert(0, job['project_root'])
    evaluation = import_module('src.steps.04_evaluation')
    fine_tuning = import_module('src.steps.05_fine_tuning.fine_tuning')
    records = fine_tuning.load_records(job['evaluation_path'])
    root = Path(job['output_dir'])
    results = []
    table_rows = []
    field_rows = []
    for index, record in enumerate(records, 1):
        case = f'case-{index:05d}'
        path = root / 'cases' / f'{case}.json'
        if not path.exists():
            continue
        saved = load_case(path, job['fingerprint'])
        item = evaluation.EvaluationItem(case, record['resume_text'], record['target_json'], record.get('category'))
        result = evaluation.EvaluationResult(
            item=item, **{key: saved[key] for key in SCORE_FIELDS},
            score_breakdown=saved['score_breakdown'], target=record['target_json'],
            model_name=f"{job['model_repo']}@{job['model_sha']}",
        )
        results.append(result)
        table_rows.append({
            'case': case, 'category': item.category,
            **{key: saved[key] for key in SCORE_FIELDS if key != 'model_metadata'},
            **saved['model_metadata'],
        })
        for entry in saved['score_breakdown']:
            if entry.get('value') is not None:
                field_rows.append({'group': entry['group'], 'field': entry['path'], 'score': entry['value']})
    if not results:
        raise RuntimeError('No saved evaluation scores are available yet.')
    metrics = evaluation.Evaluator([])._summarize(results)
    complete = len(results) == len(records)
    metrics.update({
        'expected_rows': len(records), 'completed_rows': len(results),
        'all_rows_processed': complete,
        'retryable_error_count': sum(r.status in RETRYABLE for r in results),
        'output_limit_count': sum(bool(r.model_metadata.get('hit_output_limit')) for r in results),
        'json_valid_rate': metrics['json_valid_count'] / len(results),
        'schema_valid_rate': metrics['schema_valid_count'] / len(results),
    })
    atomic_json(root / 'metrics.json', metrics)
    table = pd.DataFrame(table_rows)
    table.to_csv(root / 'per_resume.csv', index=False)
    field_table = pd.DataFrame(field_rows, columns=['group', 'field', 'score'])
    if len(field_table):
        field_table = field_table.groupby(['group', 'field'], as_index=False).agg(
            mean_score=('score', 'mean'), scored_comparisons=('score', 'count'),
        )
    field_table.to_csv(root / 'per_field.csv', index=False)
    summary = pd.DataFrame([
        {'metric': key, 'value': value} for key, value in metrics.items()
        if isinstance(value, (int, float, bool)) or value is None
    ])
    summary.to_csv(root / 'summary.csv', index=False)
    report = evaluation.EvaluationReport(metrics, results, None, [])
    if complete:
        charts = evaluation.report_results({'resume-parser — post evaluation': report})
        charts.save(root / 'charts')
    return metrics, summary, table, field_table


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    parser.add_argument('--rank', type=int, required=True)
    parser.add_argument('--world-size', type=int, required=True)
    arguments = parser.parse_args()
    try:
        worker(json.loads(Path(arguments.job).read_text()), arguments.rank, arguments.world_size)
    except Exception as error:
        # Third-party exception messages may include private resume or API data.
        print(f'Evaluation worker stopped ({type(error).__name__}). '
              'Anonymous completed cases remain saved; inspect status counts and Kaggle secrets/quota.',
              flush=True)
        sys.exit(1)
'''
worker_path = PRIVATE_ROOT / 'post_evaluation_runner.py'
worker_path.write_text(runner_source.lstrip('\n'), encoding='utf-8')
runner_spec = spec_from_file_location('post_evaluation_runner', worker_path)
post_evaluation = module_from_spec(runner_spec)
runner_spec.loader.exec_module(post_evaluation)
fine_tuning = import_module('src.steps.05_fine_tuning.fine_tuning')
evaluation = import_module('src.steps.04_evaluation')
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT not in (1, 2):
    raise RuntimeError('Use a Kaggle T4 GPU session (one or two visible GPUs).')
runtime_versions = {name: version(name) for name in (
    'torch', 'unsloth', 'unsloth_zoo', 'transformers', 'peft', 'bitsandbytes',
    'typesafe-sdk', 'pydantic', 'huggingface_hub',
)}
print('Evaluation GPUs:', GPU_COUNT, '| 4-bit | FP16')
print(json.dumps(runtime_versions, indent=2))

In [ ]:
# Cell 6 — Pin the published model and configure this post-evaluation benchmark.
HF_MODEL_REPO = 'md-zain/resume-parser'
MODEL_REVISION = 'main'  # Resolve the public model once and record its immutable commit.
TRAINING_CHECKPOINT = '4128d0e4627ee5d71f88c4e9f14efc4f147f64c2'
HF_DATASET_REPO = 'md-zain/resume-parser-dataset'
MAX_NEW_TOKENS = 2048  # Fixed before evaluation; report every output-limit hit.
OUTPUT_DIR = Path('/kaggle/working/post-evaluation')
PREVIOUS_RESULTS_DIR = None  # Optional '/kaggle/input/YOUR-PREVIOUS-OUTPUT/post-evaluation'.
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'resume-parser')
WANDB_ENTITY = os.getenv('WANDB_ENTITY')

In [ ]:
# Cell 7 — Verify the original reserved evaluation split and the training prompt/schema.
api = HfApi(token=hf_token)
model_info = api.model_info(HF_MODEL_REPO, revision=MODEL_REVISION)
MODEL_SHA = model_info.sha
manifest_path = hf_hub_download(
    HF_MODEL_REPO, filename='checkpoint/run_manifest.json', revision=TRAINING_CHECKPOINT,
    token=hf_token,
)
training_manifest = json.loads(Path(manifest_path).read_text())
dataset_revision = os.getenv('HF_DATASET_REVISION') or training_manifest.get('dataset_revision', 'main')
DATASET_SHA = api.dataset_info(HF_DATASET_REPO, revision=dataset_revision).sha
os.environ['HF_DATASET_REVISION'] = DATASET_SHA
evaluation_path = fine_tuning.download_split(
    'test', curated_dir=PRIVATE_ROOT / 'curated', repo_id=HF_DATASET_REPO,
)
evaluation_hash = hashlib.sha256(evaluation_path.read_bytes()).hexdigest()
if evaluation_hash != training_manifest['split_sha256']['test']:
    raise RuntimeError('The evaluation split changed since training. Set HF_DATASET_REVISION to the original dataset commit.')
records = fine_tuning.load_records(evaluation_path)
fine_tuning._validate_records(records, 'held-out evaluation')
heldout_ids = {str(row['resume_id']) for row in records}
if len(heldout_ids) != len(records):
    raise RuntimeError('Evaluation resume IDs are not unique. Inspect the source split before benchmarking.')
if heldout_ids.intersection(map(str, training_manifest['train_resume_ids'])):
    raise RuntimeError('Evaluation identities overlap the recorded training selection.')
prompt_hash = hashlib.sha256(fine_tuning.render_training_prompt('STATIC_PROMPT_MARKER', style='compact').encode()).hexdigest()
schema_hash = hashlib.sha256((PROJECT_ROOT / 'src/schema/resume_output.py').read_bytes()).hexdigest()
if prompt_hash != training_manifest['prompt_sha256'] or schema_hash != training_manifest['schema_sha256']:
    raise RuntimeError('The extraction prompt/schema differs from training. Restore the matching source before evaluation.')
BASE_MODEL = training_manifest['model_id']
CONTEXT_LENGTH = int(AutoConfig.from_pretrained(BASE_MODEL, token=hf_token).max_position_embeddings)
print('Reserved evaluation resumes:', len(records), '| sampling: none | truncation of inputs/targets: none')
print('Evaluation split matches training manifest:', True)
print('Model commit:', MODEL_SHA)
print('Context:', CONTEXT_LENGTH, '| maximum generated tokens:', MAX_NEW_TOKENS)
# Inputs exceeding context remain failures in the denominator; rows are not silently dropped.

In [ ]:
# Cell 8 — Stage the pinned adapter and bind saved scores to the exact benchmark.
ADAPTER_FILES = [
    'adapter_config.json', 'adapter_model*.safetensors', 'adapter_model*.bin',
    'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
    'added_tokens.json', 'vocab.json', 'merges.txt', 'chat_template.jinja', 'chat_templates/*.jinja',
]
adapter_path = snapshot_download(
    HF_MODEL_REPO, revision=MODEL_SHA, allow_patterns=ADAPTER_FILES, token=hf_token,
)
# Read the published adapter config; the GPU runners load its base in 4-bit and attach these weights.
adapter_config = json.loads((Path(adapter_path) / 'adapter_config.json').read_text())
if adapter_config.get('peft_type') != 'LORA' or adapter_config.get('base_model_name_or_path') != BASE_MODEL:
    raise RuntimeError('The public model must contain the trained LoRA adapter for the expected quantized base.')
if not list(Path(adapter_path).glob('adapter_model*.safetensors')):
    raise RuntimeError('The public model is missing trained adapter weights.')
print('Public adapter:', HF_MODEL_REPO, '| commit:', MODEL_SHA)
print('Quantized base:', BASE_MODEL, '| LoRA rank:', adapter_config['r'], '| load_in_4bit=True | FP16')
print('Each GPU loads the base and trained adapter automatically in Cell 9.')
descriptor = {
    'model_repo': HF_MODEL_REPO, 'model_sha': MODEL_SHA,
    'training_checkpoint': TRAINING_CHECKPOINT,
    'dataset_repo': HF_DATASET_REPO, 'dataset_sha': DATASET_SHA, 'evaluation_data_sha256': evaluation_hash,
    'evaluation_rows': len(records), 'prompt_sha256': prompt_hash, 'schema_sha256': schema_hash,
    'context_length': CONTEXT_LENGTH, 'max_new_tokens': MAX_NEW_TOKENS,
    'generation': 'greedy; EOS or complete JSON checked every 16 tokens; no input truncation',
    'seed': 42, 'precision': '4-bit weights / FP16 compute', 'runtime_versions': runtime_versions,
    'runner_sha256': hashlib.sha256(worker_path.read_bytes()).hexdigest(),
    'source_sha256': {
        path: hashlib.sha256((PROJECT_ROOT / path).read_bytes()).hexdigest()
        for path in (
            'src/steps/04_evaluation/evaluator.py', 'src/steps/05_fine_tuning/fine_tuning.py',
        )
    },
}
fingerprint = hashlib.sha256(json.dumps(descriptor, sort_keys=True).encode()).hexdigest()
descriptor['fingerprint'] = fingerprint
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if PREVIOUS_RESULTS_DIR:
    prior = Path(PREVIOUS_RESULTS_DIR)
    old_manifest = json.loads((prior / 'run_manifest.json').read_text())
    if old_manifest != descriptor:
        raise RuntimeError('Previous scores do not match this exact benchmark; use the original settings/source/versions.')
    for path in (prior / 'cases').glob('case-*.json'):
        post_evaluation.load_case(path, fingerprint)
        destination = OUTPUT_DIR / 'cases' / path.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copy2(path, destination)
saved_manifest = OUTPUT_DIR / 'run_manifest.json'
if saved_manifest.exists() and json.loads(saved_manifest.read_text()) != descriptor:
    raise RuntimeError('Output directory contains a different benchmark; use a fresh session.')
post_evaluation.atomic_json(saved_manifest, descriptor)
job = {
    **descriptor, 'project_root': str(PROJECT_ROOT), 'evaluation_path': str(evaluation_path),
    'adapter_path': adapter_path, 'output_dir': str(OUTPUT_DIR),
}
job_path = PRIVATE_ROOT / 'evaluation-job.json'
post_evaluation.atomic_json(job_path, job)
print('Benchmark fingerprint:', fingerprint)
print('Previously saved cases:', len(list((OUTPUT_DIR / 'cases').glob('case-*.json'))))
print('Anonymous scores:', OUTPUT_DIR)

In [ ]:
# Cell 9 — Evaluate all reserved rows across the T4s; save each result immediately.
from concurrent.futures import ThreadPoolExecutor, wait
import wandb

evaluation_run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name=f'resume-parser-post-evaluation-step-714-{fingerprint[:8]}',
    job_type='post-evaluation', group=f'post-evaluation-{fingerprint[:12]}', config=descriptor,
)

def run_worker(rank):
    child_env = os.environ.copy()
    child_env['CUDA_VISIBLE_DEVICES'] = str(rank)
    child_env['PYTHONUNBUFFERED'] = '1'
    with subprocess.Popen(
        [sys.executable, str(worker_path), '--job', str(job_path),
         '--rank', str(rank), '--world-size', str(GPU_COUNT)],
        cwd=PROJECT_ROOT, env=child_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(f'[GPU {rank}] {line}', end='', flush=True)
        return process.wait()

with ThreadPoolExecutor(max_workers=GPU_COUNT) as pool:
    futures = [pool.submit(run_worker, rank) for rank in range(GPU_COUNT)]
    remaining = set(futures)
    last_count = -1
    while remaining:
        _, remaining = wait(remaining, timeout=15)
        saved = [post_evaluation.load_case(path, fingerprint)
                 for path in (OUTPUT_DIR / 'cases').glob('case-*.json')]
        if len(saved) != last_count:
            last_count = len(saved)
            evaluation_run.log({
                'post_eval_progress/completed_cases': last_count,
                'post_eval_progress/expected_cases': len(records),
                'post_eval_progress/overall_including_failures': (
                    sum(case['overall_score'] or 0 for case in saved) / last_count if last_count else 0
                ),
            })
    worker_codes = [future.result() for future in futures]
print('Worker exit codes:', worker_codes)
# Continue to the summary even after a worker error so partial scores are visible.

In [ ]:
# Cell 10 — Display/save evaluator tables and charts; sync anonymous scores to W&B.
import pandas as pd
from IPython.display import display
try:
    metrics, summary, per_resume, per_field = post_evaluation.summarize(job)
    display(summary, per_resume, per_field)
    print('Status counts:', metrics['status_counts'])
    print('Scores/rates use 0–1; missing semantic scores remain unscored.')
    print('Benchmark complete:', metrics['all_rows_processed'])
    print('Saved outputs:', OUTPUT_DIR)
    for path in sorted(OUTPUT_DIR.glob('gpu-*.json')):
        print(path.name, json.loads(path.read_text()))
    evaluation_run.log({
        **{f'post_eval/{key}': value for key, value in metrics.items() if isinstance(value, (int, float, bool))},
        'post_eval/summary': wandb.Table(dataframe=summary),
        'post_eval/per_resume': wandb.Table(dataframe=per_resume),
        'post_eval/per_field': wandb.Table(dataframe=per_field),
    })
    evaluation_run.summary['post_eval/status_counts'] = metrics['status_counts']
finally:
    evaluation_run.finish()
if any(worker_codes) or not metrics['all_rows_processed'] or metrics['retryable_error_count']:
    raise RuntimeError('Evaluation is incomplete or has service/inference errors. Attach saved outputs and resume the same benchmark.')

## Values to check and how to continue

- **Kaggle Secrets:** `TYPESAFE_API_KEY`, `WANDB_API_KEY`; optionally `HF_TOKEN`, `GITHUB_TOKEN`,
  `WANDB_ENTITY`, `WANDB_PROJECT`, `HF_DATASET_REVISION`.
- **Cell 3:** repository URL, or `SOURCE_DIR` for attached source. Existing Steps 04/05 provide
  the evaluator, schema and training prompt; this notebook includes its Step 06 runner.
- **Cell 6:** published model/dataset IDs are already filled in for this project. The public model is loaded from `main`
  and pinned to its resolved commit for the run. There is no dataset path to set when using Hugging Face.
- **Recovery:** attach the previous saved output and set `PREVIOUS_RESULTS_DIR`; run from the top.
  Keep the same model, dataset commit, code, package versions and generation settings.
- **Reading scores:** report JSON/schema rates, programmatic and semantic means, the number fully
  scored, and the overall score including failures together. Partial results are not a full-evaluation score.
  A high output-limit count indicates incomplete outputs under this declared protocol; do not silently
  exclude those cases or present the earlier validation score as a matched evaluation comparison.
- This notebook attempts every reserved row but cannot override Kaggle's session/quota limits or TypeSafe
  service limits. Resume completed anonymous scores if a session ends. Timing depends on output lengths
  and API latency; no completion-time guarantee is made.

API references: [Unsloth inference](https://unsloth.ai/docs/basics/inference-and-deployment/unsloth-inference)
and [Transformers generation](https://huggingface.co/docs/transformers/main_classes/text_generation).